<a href="https://colab.research.google.com/github/projectgroup398-sudo/NorthStar_Section1_SQL_in_R/blob/main/NorthStar_Section1_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Required Packages

In [ ]:
# Install R and required libraries in Colab
!apt-get install -y r-base
!pip install rpy2 pandas

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
r-base is already the newest version (4.5.3-1.2204.0).
0 upgraded, 0 newly installed, 0 to remove and 6 not upgraded.


# Upload CSV Files

In [ ]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

# Load R Environment

In [ ]:
%load_ext rpy2.ipython

# Load Libraries in R

In [ ]:
%%R

install.packages("sqldf")
install.packages("dplyr")

library(sqldf)
library(dplyr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
also installing the dependencies ‘gsubfn’, ‘proto’, ‘RSQLite’, ‘chron’

trying URL 'https://cran.rstudio.com/src/contrib/gsubfn_0.7.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/proto_1.0.0.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/RSQLite_2.4.6.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/chron_2.3-62.tar.gz'
trying URL 'https://cran.rstudio.com/src/contrib/sqldf_0.4-12.tar.gz'

The downloaded source packages are in
	‘/tmp/Rtmp5ZgCkQ/downloaded_packages’
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cran.rstudio.com/src/contrib/dplyr_1.2.0.tar.gz'
Content type 'application/x-gzip' length 922393 bytes (900 KB)
downloaded 900 KB


The downloaded source packages are in
	‘/tmp/Rtmp5ZgCkQ/downloaded_packages’
Loading required package: gsubfn
Loading required package: proto
Loading required package: RSQLite

Attaching

# Import CSV Files into R Dataframes

In [ ]:
%%R

app_event <- read.csv("app_events.csv")
complaint <- read.csv("complaints.csv")
customer <- read.csv("customers.csv")
drivers <- read.csv("drivers.csv")
hubs <- read.csv("hubs.csv")
incidents <- read.csv("incidents.csv")
orders <- read.csv("orders.csv")
vehicle <- read.csv("vehicles.csv")
data_dictionary <- read.csv("data_dictionary.csv")

# Preview data
head(customer)

  customer_id age home_zone customer_type         signup_date loyalty_score
1       C0001  26     North           SME 2024-11-27 04:25:00          44.9
2       C0002  61   AIRPORT      Consumer 2025-10-28 01:04:00          55.4
3       C0003  66      East      Consumer 2025-07-02 03:23:00          75.9
4       C0004  75   CENTRAL      Consumer 2025-08-19 01:58:00          32.5
5       C0005  26 Riverside      Consumer 2025-06-03 06:02:00          55.9
6       C0006  41      WEST      Consumer 2024-03-29 13:26:00          39.9
  app_engagement_score preferred_channel account_status
1                 69.2               App         Active
2                 66.6               App         Active
3                 33.8                           Active
4                 33.0               App         Active
5                100.0               Web         Active
6                 43.3               Web         Active


# SQL Analysis 1 – Identify High Complaint Zones

Business Problem

NorthStar is experiencing customer dissatisfaction due to delivery delays and service issues. The company needs to identify which zones generate the highest complaints to prioritise operational improvements.

In [ ]:
%%R

high_complaint_zones <- sqldf("
SELECT
    o.pickup_zone,
    COUNT(c.complaint_id) AS total_complaints,
    AVG(c.resolution_days) AS avg_resolution_time
FROM orders o
LEFT JOIN complaint c
ON o.order_id = c.order_id
GROUP BY o.pickup_zone
ORDER BY total_complaints DESC
")

high_complaint_zones

   pickup_zone total_complaints avg_resolution_time
1         East               26            7.538462
2      CENTRAL               26            8.846154
3         West               25            7.960000
4        South               25            8.680000
5    Riverside               24            8.833333
6         EAST               24            6.041667
7        SOUTH               21            6.761905
8    RiverSide               21            6.476190
9        north               20            8.300000
10         Ctr               18            8.277778
11     AIRPORT               18            6.388889
12       NORTH               17            9.000000
13       North               16            9.875000
14     Central               16            6.625000
15     Airport               14            8.214286
16        WEST                9           10.888889


# SQL Analysis 2 – Customer Satisfaction vs Loyalty

Business Problem

Determine whether loyal customers experience fewer complaints. This helps evaluate whether service quality affects customer retention.

In [ ]:
%%R

customer_complaints <- sqldf("
SELECT
    cu.customer_type,
    ROUND(AVG(cu.loyalty_score),2) AS avg_loyalty,
    COUNT(co.complaint_id) AS complaint_count
FROM customer cu
LEFT JOIN complaint co
ON cu.customer_id = co.customer_id
GROUP BY cu.customer_type
")

customer_complaints

  customer_type avg_loyalty complaint_count
1      Consumer       60.09             242
2    Enterprise       60.02              28
3           SME       56.90              50


# SQL Analysis 3 – Driver Experience vs Service Demand

Business Problem

Identify whether experienced drivers are concentrated in high-demand zones.

In [ ]:
%%R

service_demand <- sqldf("
SELECT
    service_type,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(order_value),2) AS avg_order_value
FROM orders
GROUP BY service_type
ORDER BY total_orders DESC
")

service_demand

  service_type total_orders avg_order_value
1    Passenger          341           96.07
2       Parcel          308           87.62
3       Retail          297           90.01
4     Business          165           92.25
5      Medical          139           87.14


# SQL Analysis 4 – Order Demand by Service Type


Business Problem

Understand which service types generate the highest operational demand.

In [ ]:
%%R

service_demand <- sqldf("
SELECT
    service_type,
    COUNT(order_id) AS total_orders,
    ROUND(AVG(order_value),2) AS avg_order_value
FROM orders
GROUP BY service_type
ORDER BY total_orders DESC
")

service_demand

  service_type total_orders avg_order_value
1    Passenger          341           96.07
2       Parcel          308           87.62
3       Retail          297           90.01
4     Business          165           92.25
5      Medical          139           87.14


# SQL Analysis 5 – App Performance Impact on Customer Experience

Business Problem

Slow app response times may affect customer satisfaction and order completion.

In [ ]:
%%R

app_performance <- sqldf("
SELECT
    device_type,
    event_type,
    ROUND(AVG(api_latency_ms),2) AS avg_latency,
    SUM(success_flag) AS successful_events
FROM app_event
GROUP BY device_type, event_type
ORDER BY avg_latency DESC
")

app_performance

   device_type                  event_type avg_latency successful_events
1          iOS                 chat_opened      535.38                37
2          Web               payment_retry      534.30                 9
3          Web                 track_order      533.32                25
4      Android              chat_escalated      532.75                 6
5          Web              chat_escalated      526.50                 3
6      Android delivery_instruction_update      519.51                37
7          iOS delivery_instruction_update      501.38                29
8          Web                 eta_refresh      472.06                16
9          iOS                search_route      471.94                34
10         iOS               payment_retry      464.24                18
11     Android               payment_retry      460.76                23
12     Android                 eta_refresh      460.63                54
13     Android                 track_order      453

# SQL Analysis 6 – Vehicle Health Impact on Operational Reliability

Business Problem

Vehicle condition directly impacts delivery reliability. Poor battery health or maintenance issues may contribute to service delays, cancellations, or inefficient dispatch allocation.

Understanding fleet condition helps NorthStar improve preventive maintenance planning and reduce operational disruptions.

In [ ]:
%%R

vehicle_health_summary <- sqldf("
SELECT
    assigned_zone,
    vehicle_type,
    COUNT(vehicle_id) AS total_vehicles,
    ROUND(AVG(battery_health_pct),2) AS avg_battery_health,
    SUM(CASE WHEN maintenance_status = 'InRepair' THEN 1 ELSE 0 END) AS vehicles_in_repair
FROM vehicle
GROUP BY assigned_zone, vehicle_type
ORDER BY avg_battery_health ASC
")

vehicle_health_summary

   assigned_zone vehicle_type total_vehicles avg_battery_health
1        AIRPORT       Diesel              3              55.07
2           East       Diesel              1              55.90
3            Ctr       Hybrid              1              57.30
4           West     CargoVan              1              58.60
5        Central       Hybrid              1              58.90
6          South       Diesel              2              59.10
7           EAST     CargoVan              1              62.40
8          SOUTH       Hybrid              3              62.50
9           EAST       Diesel              1              62.70
10         South     CargoVan              2              62.70
11          EAST       Hybrid              2              64.60
12          WEST           EV              2              64.90
13       AIRPORT       Hybrid              4              65.17
14     Riverside     CargoVan              3              68.43
15       Central     CargoVan           

# SQL Analysis 7 – Incident Severity Impact on Resolution Time

Business Problem

Operational incidents such as route deviations, battery alerts, or missing proof of delivery increase delays and affect service reliability.

Understanding incident severity helps prioritise response processes and reduce operational risk.

In [ ]:
%%R

incident_analysis <- sqldf("
SELECT
    incident_type,
    severity,
    COUNT(incident_id) AS total_incidents,
    ROUND(AVG(resolved_hours),2) AS avg_resolution_time
FROM incidents
GROUP BY incident_type, severity
ORDER BY avg_resolution_time DESC
")

incident_analysis

      incident_type severity total_incidents avg_resolution_time
1  TemperatureIssue   Medium              13               16.32
2    RouteDeviation      Low              17               16.02
3      BatteryAlert     High               8               15.74
4  TemperatureIssue      Low               6               15.33
5      AppSyncError      Low               9               15.28
6      ProofMissing      Low              13               14.89
7    CustomerNoShow     High              11               14.60
8      VehicleFault Critical               3               14.57
9    RouteDeviation Critical               4               14.20
10   CustomerNoShow      Low              15               14.14
11   CustomerNoShow   Medium              13               14.06
12   SafetyNearMiss   Medium               6               13.28
13     VehicleFault     High              10               13.19
14   RouteDeviation     High              12               13.06
15     BatteryAlert Criti